# MPP Converter Output Reader

Read CSV output files from Azure Blob Storage using the MPP converter settings in `.env`.

In [55]:
import os
from io import BytesIO
from pathlib import Path
from typing import Dict, Optional

import pandas as pd
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv


ENV_CONNECTION_STRING = "AZURE_STORAGE_CONNECTION_STRING_MPP_CONVERTER"
ENV_CONTAINER_NAME = "AZURE_STORAGE_CONTAINER_MPP_CONVERTER_OUT"

REQUIRED_COLUMNS = ["TaskName", "StartDate", "FinishDate", "WeekOfYear"]

GATE_OUTPUT_COLUMNS = [
    "WorkPackage",
    "GateNumber",
    "StartDate",
    "FinishDate",
    "WeekOfYear",
]


def find_env_file(start_dir: Optional[Path] = None) -> Path:
    """
    Find .env file in the current directory or its parent directory.
    """
    current_dir = start_dir or Path.cwd()

    candidate_paths = [
        current_dir / ".env",
        current_dir.parent / ".env",
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Could not find .env file in {current_dir} or {current_dir.parent}"
    )


def get_mpp_output_container_client():
    """
    Create and return the Azure Blob container client for MPP converter output.
    """
    env_path = find_env_file()
    load_dotenv(env_path)

    connection_string = os.getenv(ENV_CONNECTION_STRING)
    container_name = os.getenv(ENV_CONTAINER_NAME)

    if not connection_string:
        raise ValueError(f"{ENV_CONNECTION_STRING} is not configured in .env")

    if not container_name:
        raise ValueError(f"{ENV_CONTAINER_NAME} is not configured in .env")

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    return blob_service.get_container_client(container_name)


def read_csv_blob(container_client, blob_name: str, **read_csv_kwargs) -> pd.DataFrame:
    """
    Download a CSV blob from Azure Blob Storage and return it as a DataFrame.
    """
    blob_client = container_client.get_blob_client(blob_name)
    content = blob_client.download_blob().readall()

    return pd.read_csv(BytesIO(content), **read_csv_kwargs)


def list_csv_blobs(container_client) -> list[str]:
    """
    List all CSV blob names in the container.
    """
    return [
        blob.name
        for blob in container_client.list_blobs()
        if blob.name.lower().endswith(".csv")
    ]


def validate_required_columns(df: pd.DataFrame, blob_name: str) -> None:
    """
    Validate that the required columns exist in the DataFrame.
    """
    missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]

    if missing_cols:
        raise ValueError(
            f"Blob '{blob_name}' is missing required columns: {missing_cols}"
        )


def extract_gate_rows(csv_dataframes: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Extract gate rows from multiple CSV DataFrames.

    Expected TaskName pattern examples:
        "1. Gate 1"
        "2. Gate 2"
        "10. Gate 4"
    """
    pattern = r"\b(\d+)\.\s*Gate\s+(\d+)\b"

    gate_rows = []

    for blob_name, df in csv_dataframes.items():
        validate_required_columns(df, blob_name)

        filtered = df[
            df["TaskName"].astype(str).str.contains(pattern, na=False, regex=True)
        ].copy()

        if filtered.empty:
            continue

        extracted = filtered["TaskName"].str.extract(pattern)

        # The second capture group is the gate number
        filtered["GateNumber"] = extracted[1].astype("Int64")
        filtered["WorkPackage"] = Path(blob_name).stem

        filtered = filtered[GATE_OUTPUT_COLUMNS]

        gate_rows.append(filtered)

    if not gate_rows:
        return pd.DataFrame(columns=GATE_OUTPUT_COLUMNS)

    return pd.concat(gate_rows, ignore_index=True)


def filter_by_finish_year(df: pd.DataFrame, plan_year: int) -> pd.DataFrame:
    """
    Filter gate rows by FinishDate year.
    """
    if df.empty:
        return df

    df = df.copy()

    df["FinishDate"] = pd.to_datetime(df["FinishDate"], errors="coerce")

    return df[df["FinishDate"].dt.year == plan_year]


def create_gate_summary(
    gate_rows: pd.DataFrame,
    gate_numbers: list[int] | None = None,
    plan_year: int = 2026
) -> pd.DataFrame:
    """
    Create a gate summary table:

        work_package_name | forecast_gate_1 | forecast_gate_2 | forecast_gate_3 | forecast_gate_4
    """
    if gate_numbers is None:
        gate_numbers = [1, 2, 3, 4]

    desired_cols = ["work_package_name"] + [
        f"forecast_gate_{gate}" for gate in gate_numbers
    ]

    if gate_rows.empty:
        return pd.DataFrame(columns=desired_cols)

    gate_rows = gate_rows.copy()

    # Create final forecast gate column names
    gate_rows["GateCol"] = (
        "forecast_gate_" + gate_rows["GateNumber"].astype(str)
    )

    gate_summary = (
        gate_rows
        .pivot_table(
            index="WorkPackage",
            columns="GateCol",
            values="WeekOfYear",
            aggfunc="first",
        )
        .reset_index()
    )
    gate_summary.columns.name = None

    # Rename WorkPackage to work_package
    gate_summary = gate_summary.rename(columns={"WorkPackage": "work_package_name"})
    gate_summary['plan_year'] = plan_year

    # Ensure all expected columns exist
    for col in desired_cols:
        if col not in gate_summary.columns:
            gate_summary[col] = pd.NA
    
    

    return gate_summary[desired_cols]


def build_gate_summary(plan_year: int = 2026) -> pd.DataFrame:
    """
    Full pipeline:
    1. Connect to Azure Blob container
    2. Read all CSV files
    3. Extract gate rows
    4. Filter by FinishDate year
    5. Create gate summary table
    """
    container_client = get_mpp_output_container_client()

    csv_blob_names = list_csv_blobs(container_client)

    csv_dataframes = {
        blob_name: read_csv_blob(container_client, blob_name)
        for blob_name in csv_blob_names
    }

    gate_rows = extract_gate_rows(csv_dataframes)
    gate_rows = filter_by_finish_year(gate_rows, plan_year)
    return create_gate_summary(gate_rows,plan_year=plan_year)


In [56]:
gate_summary = build_gate_summary(plan_year=2026)
gate_summary

C:\Users\liaow\AppData\Local\Temp\ipykernel_7840\3870075147.py:115: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["TaskName"].astype(str).str.contains(pattern, na=False, regex=True)
C:\Users\liaow\AppData\Local\Temp\ipykernel_7840\3870075147.py:115: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["TaskName"].astype(str).str.contains(pattern, na=False, regex=True)
C:\Users\liaow\AppData\Local\Temp\ipykernel_7840\3870075147.py:115: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["TaskName"].astype(str).str.contains(pattern, na=False, regex=True)
C:\Users\liaow\AppData\Local\Temp\ipykernel_7840\3870075147.py:115: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the gr

,work_package_name,forecast_gate_1,forecast_gate_2,forecast_gate_3,forecast_gate_4
0,Plymouth WP1 schedule final,NaN,5.0,11.0,25.0
1,Sunderland - LEVI,NaN,6.0,9.0,32.0
2,Sunderland LEVI,NaN,6.0,9.0,32.0
3,Surrey WP11 Schedule (new),NaN,15.0,21.0,35.0
4,Surrey WP12 Schedule1,NaN,15.0,24.0,38.0
5,Warrington Project Plan,3.0,19.0,23.0,46.0
